# Clasificación

In [ ]:
# Creamos un modelo base para cada uno y comprobamos cual es mejor frente a Train en validación cruzada

modelos = ["KNN","Logistic","RandomF","XGBoost","LightGBM"]
metricas = []

lr_clf = LogisticRegression(max_iter = 10000)
rf_clf = RandomForestClassifier(max_depth = 5, random_state = 42)
xgb_clf = XGBClassifier(max_depth = 5, random_state = 42)
lgb_clf = LGBMClassifier(max_depth= 5, random_state = 42, verbose = -1, n_jobs= -1)

for nombre, modelo in zip(modelos,[knn_clf, lr_clf, rf_clf, xgb_clf, lgb_clf]):
    print(f"Para {nombre}:", end = " ")
    if nombre not in modelos[0:2]:
        metrica = np.mean(cross_val_score(modelo, X_train, y_train, cv = 5, scoring = "balanced_accuracy")) 
    else:
        metrica = np.mean(cross_val_score(modelo, X_train_scaled, y_train, cv = 5, scoring = "balanced_accuracy"))
    print(metrica)
    metricas.append(metrica)

In [ ]:
# 3 formas para balancear
# - SMOTE - Oversampling 
# - Undersampling
# - Class_weight = 'balanced'

In [ ]:
smote = SMOTE(random_state=42)

# Para modelos de árbol (RF, XGBoost, LightGBM): SMOTE en el espacio original.
# Los árboles no dependen de distancias entre features, así que la escala no importa.
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Para KNN y Regresión Logística: SMOTE en el espacio escalado.
# SMOTE usa distancias KNN internamente para generar sintéticos —
# aplicarlo en el espacio normalizado garantiza que todas las features
# contribuyen proporcionalmente y los sintéticos son coherentes con el modelo.
X_train_smote_scaled, y_train_smote_scaled = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
print(len(y_train))
print(len(y_train_smote))

In [ ]:
# Visualizar distribución Original VS Smote

original_counts = y_train.value_counts()
smote_counts = y_train_smote.value_counts()

fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
primera = ax[0]
segunda = ax[1]

barplot_original = sns.barplot(x=original_counts.index, y=original_counts.values, ax=primera, palette='viridis', hue = original_counts.index, legend=False)
primera.set_title('Distribución Original')
primera.set_xlabel('Clase')
primera.set_ylabel('Cantidad')

for bar in barplot_original.patches:
    primera.annotate(f'{int(bar.get_height())}', 
                     xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), 
                     ha='center', va='bottom', fontsize=8)


barplot_smote = sns.barplot(x=smote_counts.index, y=smote_counts.values, ax=segunda, palette='viridis', hue = smote_counts.index, legend=False)
segunda.set_title('Distribución SMOTE')
segunda.set_xlabel('Clase')

for bar in barplot_smote.patches:
    segunda.annotate(f'{int(bar.get_height())}', 
                     xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()), 
                     ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


# Regresión

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# Baseline lineal: si el MAPE de LinearRegression ya es bajo,
# el problema era esencialmente lineal y los ensembles no aportan mucho.
# Si los ensembles mejoran claramente, la relación es no lineal — y tiene sentido usarlos.
lr_reg  = LinearRegression()
dt_reg  = DecisionTreeRegressor(max_depth=5, random_state=42)
rf_reg  = RandomForestRegressor(max_depth=10, random_state=42)
lgb_reg = LGBMRegressor(max_depth=10, random_state=42, verbose=-1)
xgb_reg = XGBRegressor(max_depth=10, random_state=42)

modelos_reg = {
    "Regresion Lineal":    lr_reg,
    "Decision Tree":       dt_reg,
    "Random Forest":       rf_reg,
    "LightGBM":            lgb_reg,
    "XGBoost Regressor":   xgb_reg
}

In [ ]:
for feature_set, X_train in X_train_reg_dict.items():
    print(f"Para el set {feature_set}:")
    for tipo,modelo in modelos_reg.items():
        print(f"{tipo}: ", end = " ")
        print(-np.mean(cross_val_score(modelo, X_train, y_train_reg, cv = 5, scoring = "neg_mean_absolute_percentage_error"))) # MAPE = 0.024 -> ¿bueno o malo? 2,4% de error! 
    print("******")